In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# ACC Cue Theta Power Extraction

This notebook loads cue-aligned TFRs for all eligible subjects, restricts to ACC electrodes, and extracts mean theta power from 4 to 8 Hz during the cue window.

It reuses the existing `Subject` TFR cache path through `get_subject_roi_event_spectra(...)` in `mv_regression_utils.py` and does not call `_get_smoothed_event_signal`.


In [ ]:
import os
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from membandit.io import Subject
from membandit._impl.multivariate import get_subject_roi_event_spectra

sns.set_context('talk')
sns.set_style('whitegrid')


In [ ]:
# ---- Config ----
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'

roi = 'ACC'
event = 'cue_on'
freq_range = (4.0, 8.0)
event_window = (0.0, 1.0)
condition = 'Day1'
min_trials = 10

analysis_output_root = Path(
    '/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/'
    'analysis_outputs/acc_cue_theta_power'
)
figure_output_root = analysis_output_root / 'figures'
analysis_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)

paths: Dict[str, Path] = {
    'trial_electrode_csv': analysis_output_root / 'acc_cue_theta_trial_electrode.csv',
    'subject_summary_csv': analysis_output_root / 'acc_cue_theta_subject_summary.csv',
    'audit_csv': analysis_output_root / 'acc_cue_theta_audit.csv',
    'time_subject_csv': analysis_output_root / 'acc_cue_theta_time_subject.csv',
    'time_summary_csv': analysis_output_root / 'acc_cue_theta_time_summary.csv',
    'subject_hist_png': figure_output_root / 'acc_cue_theta_subject_mean_hist.png',
    'subject_box_png': figure_output_root / 'acc_cue_theta_subject_mean_box.png',
    'electrode_violin_png': figure_output_root / 'acc_cue_theta_electrode_violin.png',
    'time_subjects_png': figure_output_root / 'acc_cue_theta_time_subjects.png',
    'time_summary_png': figure_output_root / 'acc_cue_theta_time_summary.png',
}

print('Analysis output root:', analysis_output_root)
print('Figure output root:', figure_output_root)
print('ROI:', roi)
print('Event:', event)
print('Theta range (Hz):', freq_range)
print('Event window (s):', event_window)
print('Condition:', condition)
print('Minimum aligned trials:', min_trials)


In [ ]:
# ---- Subject selection ----
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])].copy()
subjects: List[str] = sorted(subj_df['MSSMCode'].dropna().astype(str).unique().tolist())

print('Eligible subjects:', len(subjects))
print('First 10 subjects:', subjects[:10])


In [ ]:
# ---- Single-subject smoke check ----
smoke_subject = subjects[0]
smoke_subject_obj = Subject(smoke_subject, base_dir, load_dir, min_trials=min_trials, verbose=False)
smoke_result = get_subject_roi_event_spectra(
    subject_obj=smoke_subject_obj,
    roi=roi,
    event=event,
    event_window=event_window,
    freq_range=freq_range,
    condition=condition,
    min_trials=min_trials,
)

print('Smoke subject:', smoke_subject)
print('spectral_power shape:', smoke_result['spectral_power'].shape)
print('n_trials:', smoke_result['n_trials'])
print('n_electrodes:', len(smoke_result['electrode_labels']))
print('n_freqs:', len(smoke_result['freqs']))
print('freqs:', smoke_result['freqs'])
assert smoke_result['spectral_power'].ndim == 3
assert smoke_result['spectral_power'].shape[0] == smoke_result['n_trials']
assert smoke_result['spectral_power'].shape[1] == len(smoke_result['electrode_labels'])
assert smoke_result['spectral_power'].shape[2] == len(smoke_result['freqs'])
assert len(smoke_result['freqs']) > 0
assert float(np.nanmin(smoke_result['freqs'])) >= freq_range[0]
assert float(np.nanmax(smoke_result['freqs'])) <= freq_range[1]


In [ ]:
# ---- Extract trial/electrode theta power for all subjects ----
trial_rows = []
summary_rows = []
audit_rows = []

for subject in subjects:
    try:
        subject_obj = Subject(subject, base_dir, load_dir, min_trials=min_trials, verbose=False)
        result = get_subject_roi_event_spectra(
            subject_obj=subject_obj,
            roi=roi,
            event=event,
            event_window=event_window,
            freq_range=freq_range,
            condition=condition,
            min_trials=min_trials,
        )

        spectral_power = np.asarray(result['spectral_power'], dtype=np.float64)
        theta_mean = np.nanmean(spectral_power, axis=-1)
        trial_indices_zero_based = np.asarray(result['trial_indices'], dtype=np.int64)
        trial_numbers_one_based = trial_indices_zero_based + 1
        electrode_labels = np.asarray(result['electrode_labels'], dtype=object)
        electrode_hemi = np.asarray(result['electrode_hemi'], dtype=object)
        electrode_region = np.asarray(result['electrode_region'], dtype=object)
        freqs = np.asarray(result['freqs'], dtype=np.float64)

        for trial_pos, trial_idx0 in enumerate(trial_indices_zero_based):
            for elec_pos, elec_label in enumerate(electrode_labels):
                trial_rows.append({
                    'subject': subject,
                    'roi': roi,
                    'event': event,
                    'condition': condition,
                    'trial_column': result['trial_column'],
                    'trial_position': int(trial_pos),
                    'trial_index_zero_based': int(trial_idx0),
                    'trial_number': int(trial_numbers_one_based[trial_pos]),
                    'electrode_position': int(elec_pos),
                    'electrode_label': str(elec_label),
                    'electrode_region': str(electrode_region[elec_pos]),
                    'electrode_hemi': electrode_hemi[elec_pos],
                    'theta_mean_power': float(theta_mean[trial_pos, elec_pos]),
                    'n_freqs_retained': int(len(freqs)),
                    'freq_min_hz': float(np.nanmin(freqs)),
                    'freq_max_hz': float(np.nanmax(freqs)),
                    'event_window_start_s': float(result['event_window'][0]),
                    'event_window_end_s': float(result['event_window'][1]),
                })

        summary_rows.append({
            'subject': subject,
            'roi': roi,
            'event': event,
            'condition': condition,
            'n_trials': int(result['n_trials']),
            'n_electrodes': int(len(electrode_labels)),
            'n_freqs': int(len(freqs)),
            'n_dropped_alignment': int(result['n_dropped_alignment']),
            'theta_mean_power_subject': float(np.nanmean(theta_mean)),
            'theta_median_power_subject': float(np.nanmedian(theta_mean)),
            'theta_std_power_subject': float(np.nanstd(theta_mean)),
        })

        audit_rows.append({
            'subject': subject,
            'status': 'ok',
            'error': '',
            'roi': roi,
            'event': event,
            'condition': condition,
            'n_trials': int(result['n_trials']),
            'n_electrodes': int(len(electrode_labels)),
            'n_freqs': int(len(freqs)),
            'n_dropped_alignment': int(result['n_dropped_alignment']),
            'theta_mean_power_subject': float(np.nanmean(theta_mean)),
        })
    except Exception as exc:
        audit_rows.append({
            'subject': subject,
            'status': 'error',
            'error': str(exc),
            'roi': roi,
            'event': event,
            'condition': condition,
            'n_trials': np.nan,
            'n_electrodes': np.nan,
            'n_freqs': np.nan,
            'n_dropped_alignment': np.nan,
            'theta_mean_power_subject': np.nan,
        })

trial_electrode_df = pd.DataFrame(trial_rows)
subject_summary_df = pd.DataFrame(summary_rows).sort_values('subject').reset_index(drop=True)
audit_df = pd.DataFrame(audit_rows).sort_values(['status', 'subject']).reset_index(drop=True)

print('Successful subjects:', int((audit_df['status'] == 'ok').sum()))
print('Failed subjects:', int((audit_df['status'] == 'error').sum()))
print('Trial/electrode rows:', len(trial_electrode_df))
display(audit_df.head(10))


## Time-Resolved Theta Across Subjects

These cells preserve cue timepoints, average 4 to 8 Hz theta within each subject, and then plot the across-subject mean with SEM error bands.


In [ ]:
# ---- Build time-resolved theta traces across subjects ----
time_subject_rows = []

for subject in subjects:
    try:
        subject_obj = Subject(subject, base_dir, load_dir, min_trials=min_trials, verbose=False)
        roi_electrodes = (
            subject_obj.elec_df[subject_obj.elec_df.salman_region == roi]['label']
            .astype(str)
            .tolist()
        )
        if len(roi_electrodes) == 0:
            raise ValueError(f'No electrodes found in ROI {roi} for subject {subject}')

        tfr_obj = subject_obj._ensure_tfr_loaded(event)
        tfr = tfr_obj[0] if isinstance(tfr_obj, list) else tfr_obj
        tfr = tfr.copy().pick(roi_electrodes)
        picked_labels = [str(ch) for ch in tfr.ch_names]
        if len(picked_labels) == 0:
            raise ValueError(f'No picked channels remain for ROI {roi} in {subject}')

        freqs = np.asarray(tfr.freqs, dtype=np.float64)
        times = np.asarray(tfr.times, dtype=np.float64)
        freq_mask = (freqs >= float(freq_range[0])) & (freqs <= float(freq_range[1]))
        time_mask = (times >= float(event_window[0])) & (times <= float(event_window[1]))
        if int(np.sum(freq_mask)) == 0:
            raise ValueError(f'No frequencies retained for range {freq_range}')
        if int(np.sum(time_mask)) == 0:
            raise ValueError(f'No timepoints retained for window {event_window}')

        subj_combined_df = subject_obj.combined_df[
            (subject_obj.combined_df.participant == subject) &
            (subject_obj.combined_df.condition == condition)
        ].copy()
        if len(subj_combined_df) == 0:
            raise ValueError(f'no_{condition.lower()}_trials_for_{subject}')

        trial_col = 'trials_mem'
        if trial_col not in subj_combined_df.columns:
            raise ValueError(f"Missing trial column '{trial_col}' for {subject}")

        subj_combined_df = subj_combined_df.dropna(subset=[trial_col]).copy()
        if len(subj_combined_df) == 0:
            raise ValueError(f'no_valid_{trial_col}_rows_for_{subject}')

        trial_vals = pd.to_numeric(subj_combined_df[trial_col], errors='coerce').to_numpy(dtype=np.float64)
        valid = (
            np.isfinite(trial_vals) &
            ((trial_vals - 1) >= 0) &
            ((trial_vals - 1) < tfr.data.shape[0])
        )
        if int(np.sum(valid)) < int(min_trials):
            raise ValueError(
                f'Insufficient aligned trials for {subject}, ROI={roi}, event={event}: '
                f"{int(np.sum(valid))} < {int(min_trials)}"
            )

        trial_ix = trial_vals[valid].astype(np.int64) - 1
        theta_time = np.nanmean(
            tfr.data[trial_ix][:, :, freq_mask, :][:, :, :, time_mask],
            axis=2,
        ).astype(np.float64)
        subject_timecourse = np.nanmean(theta_time, axis=(0, 1))
        time_values = times[time_mask].astype(np.float64)

        for time_idx, time_s in enumerate(time_values):
            time_subject_rows.append({
                'subject': subject,
                'roi': roi,
                'event': event,
                'condition': condition,
                'time_idx': int(time_idx),
                'time_s': float(time_s),
                'theta_mean_power_time': float(subject_timecourse[time_idx]),
                'n_trials': int(len(trial_ix)),
                'n_electrodes': int(len(picked_labels)),
            })
    except Exception:
        continue

time_subject_df = pd.DataFrame(time_subject_rows)
if len(time_subject_df) > 0:
    time_summary_df = (
        time_subject_df
        .groupby(['time_idx', 'time_s'], as_index=False)
        .agg(
            theta_mean_power=('theta_mean_power_time', 'mean'),
            theta_sem_power=('theta_mean_power_time', 'sem'),
            theta_std_power=('theta_mean_power_time', 'std'),
            n_subjects=('subject', 'nunique'),
        )
        .sort_values('time_idx')
        .reset_index(drop=True)
    )
else:
    time_summary_df = pd.DataFrame(
        columns=['time_idx', 'time_s', 'theta_mean_power', 'theta_sem_power', 'theta_std_power', 'n_subjects']
    )

print('Time-resolved subject rows:', len(time_subject_df))
print('Time-resolved summary rows:', len(time_summary_df))
if len(time_summary_df) > 0:
    display(time_summary_df.head())


In [ ]:
# ---- Save outputs and basic audit ----
trial_electrode_df.to_csv(paths['trial_electrode_csv'], index=False)
subject_summary_df.to_csv(paths['subject_summary_csv'], index=False)
audit_df.to_csv(paths['audit_csv'], index=False)
time_subject_df.to_csv(paths['time_subject_csv'], index=False)
time_summary_df.to_csv(paths['time_summary_csv'], index=False)

print('Saved trial/electrode table:', paths['trial_electrode_csv'])
print('Saved subject summary table:', paths['subject_summary_csv'])
print('Saved audit table:', paths['audit_csv'])
print('Saved time-resolved subject table:', paths['time_subject_csv'])
print('Saved time-resolved summary table:', paths['time_summary_csv'])

if len(trial_electrode_df) > 0:
    print('\nTrial/electrode theta summary:')
    display(trial_electrode_df['theta_mean_power'].describe().to_frame().T)

if len(subject_summary_df) > 0:
    print('\nSubject theta summary:')
    display(subject_summary_df['theta_mean_power_subject'].describe().to_frame().T)

if len(time_subject_df) > 0:
    print('\nTime-resolved subject trace summary:')
    display(time_subject_df['theta_mean_power_time'].describe().to_frame().T)

print('\nAudit status counts:')
display(audit_df['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='n_subjects'))

error_df = audit_df[audit_df['status'] == 'error'].copy()
if len(error_df) > 0:
    print('\nFailed subjects:')
    display(error_df[['subject', 'error']])


In [ ]:
# ---- Plot subject, time-resolved, and electrode distributions ----
if len(subject_summary_df) == 0:
    print('No successful subject summaries available for plotting.')
else:
    fig_hist, ax_hist = plt.subplots(figsize=(8, 5))
    sns.histplot(subject_summary_df['theta_mean_power_subject'], bins=20, kde=True, ax=ax_hist)
    ax_hist.set_title('ACC cue theta mean power across subjects')
    ax_hist.set_xlabel('Subject mean theta power')
    fig_hist.tight_layout()
    fig_hist.savefig(paths['subject_hist_png'], dpi=200, bbox_inches='tight')
    plt.show()

    fig_box, ax_box = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=subject_summary_df, y='theta_mean_power_subject', ax=ax_box)
    sns.stripplot(data=subject_summary_df, y='theta_mean_power_subject', color='black', alpha=0.65, size=4, ax=ax_box)
    ax_box.set_title('Subject mean ACC cue theta power')
    ax_box.set_ylabel('Theta mean power')
    fig_box.tight_layout()
    fig_box.savefig(paths['subject_box_png'], dpi=200, bbox_inches='tight')
    plt.show()

if len(time_subject_df) > 0:
    fig_time_subjects, ax_time_subjects = plt.subplots(figsize=(10, 6))
    for subject, subject_df in time_subject_df.groupby('subject'):
        ax_time_subjects.plot(
            subject_df['time_s'],
            subject_df['theta_mean_power_time'],
            alpha=0.25,
            linewidth=1.0,
        )
    ax_time_subjects.set_title('ACC cue theta power time courses by subject')
    ax_time_subjects.set_xlabel('Time from cue onset (s)')
    ax_time_subjects.set_ylabel('Mean theta power')
    fig_time_subjects.tight_layout()
    fig_time_subjects.savefig(paths['time_subjects_png'], dpi=200, bbox_inches='tight')
    plt.show()

if len(time_summary_df) > 0:
    fig_time_summary, ax_time_summary = plt.subplots(figsize=(10, 6))
    ax_time_summary.plot(
        time_summary_df['time_s'],
        time_summary_df['theta_mean_power'],
        color='tab:blue',
        linewidth=2,
        label='Mean across subjects',
    )
    ax_time_summary.fill_between(
        time_summary_df['time_s'],
        time_summary_df['theta_mean_power'] - time_summary_df['theta_sem_power'].fillna(0.0),
        time_summary_df['theta_mean_power'] + time_summary_df['theta_sem_power'].fillna(0.0),
        color='tab:blue',
        alpha=0.2,
        label='SEM across subjects',
    )
    ax_time_summary.axvline(0.0, color='black', linestyle='--', linewidth=1)
    ax_time_summary.set_title('ACC cue theta power time-resolved mean +/- SEM')
    ax_time_summary.set_xlabel('Time from cue onset (s)')
    ax_time_summary.set_ylabel('Mean theta power')
    ax_time_summary.legend(frameon=False)
    fig_time_summary.tight_layout()
    fig_time_summary.savefig(paths['time_summary_png'], dpi=200, bbox_inches='tight')
    plt.show()

if len(trial_electrode_df) > 0:
    electrode_plot_df = (
        trial_electrode_df
        .groupby(['subject', 'electrode_label', 'electrode_hemi'], as_index=False)['theta_mean_power']
        .mean()
        .sort_values(['subject', 'electrode_label'])
        .reset_index(drop=True)
    )
    if len(electrode_plot_df) <= 300:
        fig_violin, ax_violin = plt.subplots(figsize=(12, 6))
        sns.violinplot(data=electrode_plot_df, x='electrode_hemi', y='theta_mean_power', inner='box', ax=ax_violin)
        sns.stripplot(data=electrode_plot_df, x='electrode_hemi', y='theta_mean_power', color='black', alpha=0.45, size=3, ax=ax_violin)
        ax_violin.set_title('Mean ACC cue theta power by electrode hemisphere')
        fig_violin.tight_layout()
        fig_violin.savefig(paths['electrode_violin_png'], dpi=200, bbox_inches='tight')
        plt.show()
    else:
        print('Skipping per-electrode violin plot because the aggregated table is too large:', len(electrode_plot_df))
